# DRF Permissions & Session Authentication

## What are Permissions?

Permissions answer the question: *Is this user allowed to perform this action?*

They are checked after authentication. If a permission check fails, DRF returns `403 Forbidden` (or `401 Unauthorized` for unauthenticated requests).


## Built-in Permission Classes

| Class | Behavior |
|-------|----------|
| `AllowAny` | Open to everyone |
| `IsAuthenticated` | Must be logged in |
| `IsAdminUser` | Must be a staff/admin user |
| `IsAuthenticatedOrReadOnly` | Read for all, write for logged-in users |

### Setting Permissions

Per view:
```python
permission_classes = [IsAuthenticated]
```

Globally in `settings.py`:
```python
REST_FRAMEWORK = {
    'DEFAULT_PERMISSION_CLASSES': ['rest_framework.permissions.IsAuthenticated'],
}
```


## Custom Permissions

```python
# banking/permissions.py
from rest_framework.permissions import BasePermission, SAFE_METHODS

class IsOwner(BasePermission):
    message = "You must be the owner of this account."

    def has_object_permission(self, request, view, obj):
        return getattr(obj, 'owner_id', None) == getattr(request.user, 'id', None)

class IsOwnerOrReadOnly(BasePermission):
    message = "Only the owner may modify this account."

    def has_object_permission(self, request, view, obj):
        if request.method in SAFE_METHODS:
            return True
        return getattr(obj, 'owner_id', None) == getattr(request.user, 'id', None)
```

- `has_permission()` — checked for every request before the view runs.
- `has_object_permission()` — checked when `get_object()` is called (detail views).


## Applying Permissions to a ViewSet

```python
from rest_framework.viewsets import ReadOnlyModelViewSet
from rest_framework.mixins import CreateModelMixin
from rest_framework.permissions import IsAuthenticated, IsAdminUser
from rest_framework.decorators import action
from rest_framework.response import Response
from rest_framework import status

class BankAccountViewSet(CreateModelMixin, ReadOnlyModelViewSet):
    queryset = BankAccount.objects.all()
    serializer_class = BankAccountSerializer
    permission_classes = [IsAuthenticated, IsOwnerOrReadOnly]

    def get_queryset(self):
        if self.request.user.is_staff:
            return super().get_queryset()
        return super().get_queryset().filter(owner=self.request.user)

    def perform_create(self, serializer):
        serializer.save(owner=self.request.user)

    @action(methods=['GET'], detail=False, permission_classes=[IsAuthenticated, IsAdminUser])
    def admin_list(self, request):
        qs = BankAccount.objects.all()
        return Response(self.get_serializer(qs, many=True).data)

    @action(methods=['POST'], detail=True, url_path='withdraw',
            permission_classes=[IsAuthenticated, IsOwner])
    def withdraw_from_account(self, request, pk=None):
        account = self.get_object()
        withdraw_req_s = WithdrawRequestSerializer(data=request.data)
        withdraw_req_s.is_valid(raise_exception=True)
        amount = withdraw_req_s.validated_data['amount']
        if account.balance < amount:
            return Response({'message': f'Balance is lower than {amount}'}, status=status.HTTP_400_BAD_REQUEST)
        account.balance -= amount
        account.save()
        return Response(self.get_serializer(account).data)
```


## Session Authentication in DRF

DRF's `SessionAuthentication` reuses Django's login session, so users who have logged in via a Django view are automatically authenticated in the API.

```python
# settings.py
REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework.authentication.SessionAuthentication',
    ],
    'DEFAULT_PERMISSION_CLASSES': [
        'rest_framework.permissions.IsAuthenticated',
    ],
}
```

**Note:** Session authentication requires CSRF tokens for unsafe methods (POST, PUT, DELETE) when called from a browser.


## Testing Permissions

```python
from django.contrib.auth.models import User
from rest_framework.test import APITestCase
from rest_framework import status
from banking.models import BankAccount

class AccountPermissionTests(APITestCase):
    def setUp(self):
        self.alice = User.objects.create_user(username='alice', password='alicepw')
        self.bob = User.objects.create_user(username='bob', password='bobpw')
        self.admin = User.objects.create_user(username='admin', password='adminpw', is_staff=True)
        self.alice_acc = BankAccount.objects.create(acc_number='A-001', balance=500, owner=self.alice)
        self.bob_acc = BankAccount.objects.create(acc_number='B-001', balance=900, owner=self.bob)

    def test_list_shows_only_own_accounts(self):
        self.client.login(username='alice', password='alicepw')
        resp = self.client.get('/api/accounts/')
        self.assertEqual(resp.status_code, 200)
        self.assertEqual(len(resp.data), 1)

    def test_owner_can_withdraw(self):
        self.client.login(username='alice', password='alicepw')
        resp = self.client.post(f'/api/accounts/{self.alice_acc.pk}/withdraw/', {'amount': '100.00'}, format='json')
        self.assertEqual(resp.status_code, 200)

    def test_cannot_withdraw_from_other_account(self):
        self.client.login(username='alice', password='alicepw')
        resp = self.client.post(f'/api/accounts/{self.bob_acc.pk}/withdraw/', {'amount': '50.00'}, format='json')
        self.assertEqual(resp.status_code, status.HTTP_403_FORBIDDEN)

    def test_non_admin_cannot_access_admin_list(self):
        self.client.login(username='alice', password='alicepw')
        resp = self.client.get('/api/accounts/admin_list/')
        self.assertEqual(resp.status_code, status.HTTP_403_FORBIDDEN)
```


## Summary

- Permissions check whether an authenticated user is allowed to perform an action.
- Built-ins: `AllowAny`, `IsAuthenticated`, `IsAdminUser`, `IsAuthenticatedOrReadOnly`.
- Custom permissions extend `BasePermission`; use `has_object_permission()` for object-level checks.
- Set permissions globally in `settings.py` or per view with `permission_classes = [...]`.
- Mix permission classes: global `IsAuthenticated` + per-action `IsOwner` or `IsAdminUser`.
- `SessionAuthentication` reuses Django's session; CSRF is required for unsafe methods.
- Always test both allowed and denied scenarios in permission tests.
